# Création de la base de données

## Les données utilisées

Dans le cadre de ce projet, l'objectif est d'évaluer l'efficacité réelle du Pass Culture. Pour ce faire, nous utilisons différentes bases de données.

In [13]:
import pandas as pd
from functions import *

## Basilic

la Base des lieux et équipements culturels (Basilic) est une base de données réalisée par agrégation de différentes sources : bases de la direction générale des patrimoines et de l’architecture, de la direction générale de la création artistique, de la direction générale des médias et des industries culturelles, de la délégation générale à la transmission, aux territoires et à la démocratie culturelle, du Centre national du cinéma et de l'image animée, du Centre national du livre, du Centre national des arts du cirque, de la rue et du théâtre (Artcena), de la Médiathèque du patrimoine et de la photographie.

Elle porte sur le champ de la France entière, et va nous permettre de trouver toutes les infrastructures culturelles.

In [14]:
# Importation des données

df_brut_basilic = importdata("basilic_29072026.csv", ";")
infosbase(df_brut_basilic)

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 86366
Nombre de colonnes  : 54

NOMS DES COLONNES ET TYPES
Nom                                         object
Adresse                                     object
Complement Adresse                         float64
Code Postal                                 object
libelle_geographique                        object
code_insee                                  object
Code Insee Arrondt                          object
Identifiant origine                         object
Type équipement ou lieu                     object
Label et appellation                        object
Région                                      object
Domaine                                     object
Archéologie détail                          object
Adresse postale                             object
Département                                 object
Précision équipement                        object
N_Département                               object
N_Région               

/home/onyxia/work/Evaluation-Politiques-Publiques-Culturelles/functions.py:8: DtypeWarning: Columns (3,12,20,21,22,28,31,43) have mixed types. Specify dtype option on import or set low_memory=False.
  return (pd.read_csv(path, sep=separateur_csv, encoding="utf-8"))


Voici maintenant les informations générales sur la base, préalables au nettoyage.

In [15]:
print("\n" + "=" * 60)
print("VALEURS MANQUANTES PAR COLONNE")
print("=" * 60)
print(df_brut_basilic.isna().sum())


VALEURS MANQUANTES PAR COLONNE
Nom                                            0
Adresse                                    31397
Complement Adresse                         86366
Code Postal                                    9
libelle_geographique                           0
code_insee                                     0
Code Insee Arrondt                            85
Identifiant origine                         3013
Type équipement ou lieu                        0
Label et appellation                       30483
Région                                        12
Domaine                                        0
Archéologie détail                         85804
Adresse postale                                0
Département                                   12
Précision équipement                       65851
N_Département                                  9
N_Région                                      12
Fonction_1                                  7372
Fonction_2                           

#### Choix des variables

Maintenant, nous allons nettoyer la base, pour ne garder que les variables qui sont pertinentes pour notre sujet.

Ainsi, nous allons garder quelques variables permettant d'identifier l'infrastructure de manière unique ("Nom", "libelle_geographique","Département","Région",  "code_insee", "Type équipement ou lieu","Nombre_ecrans", "Nombre_de_salles_de_theatre").

In [16]:
df_basilic = df_brut_basilic[["Nom", 
    "libelle_geographique",
    "Département",
    "Région",  
    "code_insee", 
    "Type équipement ou lieu",
    "Nombre_ecrans", 
    "Nombre_de_salles_de_theatre"]]

Nous allons maintenant transformer la base pour que la granularité ne soit pas au niveau du batiment, mais au niveau de la ville : nous aurons ainsi une ligne par commune, et pour chaque type d'équipement, le nombre d'établissements, et le cas échéant, le nombre total de salles (cinéma, théâtre).

La première étape est de vérifier la colonne à partir de laquelle nous allons compter le nombre de bâtiments.

In [17]:
liste_etablissements=df_basilic["Type équipement ou lieu"].unique()
print(liste_etablissements)

['Monument' 'Bibliothèque' "Centre d'art" 'Centre de création artistique'
 'Cinéma' 'Papeterie et maisons de la presse' 'Centre culturel' 'Scène'
 'Espace protégé' 'Théâtre' 'Musée' 'Centre de création musicale'
 'Conservatoire' "Établissement d'enseignement supérieur" 'Parc et jardin'
 'Librairie' 'Lieu de mémoire' "Service d'archives" 'Lieu archéologique'
 'Opéra']


Maintenant, nous pouvons compter le nombre de bâtiment par ville, pour chaque type et au total.

In [18]:
# Agrégation par commune
label_dummies = pd.get_dummies(df_basilic["Type équipement ou lieu"])
agg_labels = label_dummies.groupby(df_basilic["code_insee"]).sum()

# Agrégats numériques
agg_numeriques = df_basilic.groupby("code_insee").agg(
    nb_ecrans_total=("Nombre_ecrans", "sum"),
    nb_salles_theatre_total=("Nombre_de_salles_de_theatre", "sum"),
    nb_total_etablissements=("Nom", "count")
)

# Infos communales
group_keys = ["code_insee", "libelle_geographique", "Département", "Région"]
infos_commune = df_basilic[group_keys].drop_duplicates(subset="code_insee").set_index("code_insee")

# Fusion finale
df_communes = infos_commune.join([agg_labels, agg_numeriques]).reset_index()

In [19]:
# Informations sur la base de données

infosbase(df_communes)

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 22084
Nombre de colonnes  : 27

NOMS DES COLONNES ET TYPES
code_insee                                 object
libelle_geographique                       object
Département                                object
Région                                     object
Bibliothèque                                int64
Centre culturel                             int64
Centre d'art                                int64
Centre de création artistique               int64
Centre de création musicale                 int64
Cinéma                                      int64
Conservatoire                               int64
Espace protégé                              int64
Librairie                                   int64
Lieu archéologique                          int64
Lieu de mémoire                             int64
Monument                                    int64
Musée                                       int64
Opéra                                   

A ce stade, la base de données recense donc de manière synthétique l'ensemble des équipements disponibles pour chaque commune. Nous allons maintenant y ajouter des informations complémentaires telles que des informations démographiques.

## Démographie et âges (INSEE)

Nous importons maintenant la base de données de l'INSEE permettant d'obtenir la population des communes par tranche d'âge de cinq ans et par sexe. Le but sera de joindre cette base avec les données issues de la base Basilic afin d'arriver à obtenir la densité d'équipements culturels pas habitant.

Dans un premier temps, nous importons les données. Comme pour la base Basilic, elles ont été téléchargées (le 30 juillet 2026) et stockées localement.

In [20]:
# Importation des données
df_brut_pop = importdata("DS_RP_TD_POPULATION_AGEHARSEX_PRINC_2023_data.csv", ";")

/home/onyxia/work/Evaluation-Politiques-Publiques-Culturelles/functions.py:8: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  return (pd.read_csv(path, sep=separateur_csv, encoding="utf-8"))


In [21]:
# Nettoyage de la base : seulement les variables qui nous intéressent et la population qui nous intéresse
df_pop = df_brut_pop[
    (df_brut_pop['AGE'].isin(['Y15T19', 'Y20T24'])) & (df_brut_pop['GEO_OBJECT'] == 'COM')
]
df_pop = df_pop[["GEO",
    "AGE",
    "HAR",
    "SEX",
    "FREQ",
    "OBS_STATUS",
    "OBS_VALUE"]]
infosbase(df_pop)

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 2091480
Nombre de colonnes  : 7

NOMS DES COLONNES ET TYPES
GEO            object
AGE            object
HAR            object
SEX            object
FREQ           object
OBS_STATUS     object
OBS_VALUE     float64
dtype: object


## Revenus et niveau de vie (INSEE - Filosofi)

Nous allons maintenant importer les données de revenu et de niveau de vie qui pourront servir de variables de contrôle ou d'instrument dans l'analyse de l'influence du Pass Culture sur les pratiques culturelles des jeunes.

In [42]:
df_brut_filosofi = importdata("DS_FILOSOFI_CC_2023_data.csv", ";")
df_filosofi = df_brut_filosofi[
    (df_brut_filosofi["GEO_OBJECT"] == "COM")
    & (df_brut_filosofi["FILOSOFI_MEASURE"].isin(["MED_SL", "PR_MD60"]))
]
df_filosofi = df_filosofi[[
    "GEO",
    "FILOSOFI_MEASURE",
    "OBS_VALUE"
]]
#on passe d'un format court à un format long avec la fonction pivot
df_filosofi = df_filosofi.pivot_table(
    index="GEO",
    columns="FILOSOFI_MEASURE",
    values="OBS_VALUE"
).reset_index()
df_filosofi.columns.name = None  # nettoie l'attribut résiduel
infosbase(df_filosofi)
print(df_filosofi.head())


/home/onyxia/work/Evaluation-Politiques-Publiques-Culturelles/functions.py:8: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  return (pd.read_csv(path, sep=separateur_csv, encoding="utf-8"))


DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 30815
Nombre de colonnes  : 3

NOMS DES COLONNES ET TYPES
GEO         object
MED_SL     float64
PR_MD60    float64
dtype: object
    GEO   MED_SL  PR_MD60
0  1001  28270.0      NaN
1  1002  28140.0      NaN
2  1004  24210.0     18.0
3  1005  28120.0      NaN
4  1007  28030.0      NaN


## Niveau de diplôme, éducation (INSEE)

Enfin, il s'agit d'importer les données sur le niveau de diplôme des individus, au niveau communal afin de pouvoir le comparer aux autres variables obtenues dans les autres bases de données. On choisit ici de ne garder que les données de l'année 2023.

In [24]:
# On importe les données pour le niveau communal
df_brut_diplome = importdata("DS_RP_DIPLOMES_PRINC_2023_data.csv", ";")
df_diplome = df_brut_diplome[(df_brut_diplome["GEO_OBJECT"] == "COM") & (df_brut_diplome["SEX"] == "_T")
& (df_brut_diplome["TIME_PERIOD"] == 2023)]
df_diplome = df_diplome[[
    "GEO",
    "EDUC",
    "FREQ",
    "OBS_STATUS",
    "OBS_VALUE"
]]
infosbase(df_diplome)

/home/onyxia/work/Evaluation-Politiques-Publiques-Culturelles/functions.py:8: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  return (pd.read_csv(path, sep=separateur_csv, encoding="utf-8"))


DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 313722
Nombre de colonnes  : 5

NOMS DES COLONNES ET TYPES
GEO            object
EDUC           object
FREQ           object
OBS_STATUS     object
OBS_VALUE     float64
dtype: object


Maintenant, nous allons sélectionner la population qui nous intéresse, et donner à la base un format qui nous permettra de la joindre aux autres bases préparées en amont.